# 8.2 Edit distance & string algorithms

Edit distance turns spelling disagreement into the cheapest sequence of edits, so fuzzy matching becomes a table you can audit instead of a guess.

Dynamic programming fills a grid of prefix-to-prefix repair costs. The result is useful for typos and OCR, but the cost model and threshold must match the domain because surface closeness is not semantic closeness.

Save a copy to Drive to edit.

In [ ]:

import math
import random
import re
import unicodedata
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

random.seed(8)
np.random.seed(8)


def exact_accuracy(predictions, expected):
    correct = 0
    for pred, gold in zip(predictions, expected):
        if pred == gold:
            correct += 1
    return correct / max(1, len(expected))


def cosine(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def tokenize_doc(doc):
    return re.findall(r"[a-z0-9]+", doc.casefold())


def show_matrix(ax, matrix, title, xlabels=None, ylabels=None):
    image = ax.imshow(matrix, aspect="auto", cmap="viridis")
    ax.set_title(title)
    if xlabels is not None:
        ax.set_xticks(range(len(xlabels)))
        ax.set_xticklabels(xlabels, rotation=45, ha="right")
    if ylabels is not None:
        ax.set_yticks(range(len(ylabels)))
        ax.set_yticklabels(ylabels)
    return image


## The concept, built once (D1): dynamic-programming edit distance

For source $a_1,\dots,a_m$ and target $b_1,\dots,b_n$, the table entry $D_{i,j}$ follows $$D_{i,j}=\min\{D_{i-1,j}+1,\;D_{i,j-1}+1,\;D_{i-1,j-1}+\mathbf{1}[a_i\ne b_j]\}.$$ Boundary values $D_{i,0}=i$ and $D_{0,j}=j$ price deletion and insertion from the empty prefix.

In [ ]:

def edit_distance(a, b, costs=None):
    if costs is None:
        costs = {"insert": 1, "delete": 1, "substitute": 1}
    m = len(a)
    n = len(b)
    table = np.zeros((m + 1, n + 1), dtype=int)
    back = [[None for j in range(n + 1)] for i in range(m + 1)]
    for i in range(1, m + 1):
        table[i, 0] = table[i - 1, 0] + costs["delete"]
        back[i][0] = "delete"
    for j in range(1, n + 1):
        table[0, j] = table[0, j - 1] + costs["insert"]
        back[0][j] = "insert"
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            sub_cost = 0 if a[i - 1] == b[j - 1] else costs["substitute"]
            choices = [
                (table[i - 1, j] + costs["delete"], "delete"),
                (table[i, j - 1] + costs["insert"], "insert"),
                (table[i - 1, j - 1] + sub_cost, "match" if sub_cost == 0 else "substitute"),
            ]
            best_cost, best_op = min(choices, key=lambda item: item[0])
            table[i, j] = best_cost
            back[i][j] = best_op
    return int(table[m, n]), table, back

lesson_distance, lesson_table, lesson_back = edit_distance("cat", "cut")
expected_table = np.array([
    [0, 1, 2, 3],
    [1, 0, 1, 2],
    [2, 1, 1, 2],
    [3, 2, 2, 1],
])

assert lesson_distance == 1
assert np.array_equal(lesson_table, expected_table)
assert list(lesson_table[:, 0]) == [0, 1, 2, 3]


The same helper can rank candidates by minimum repair cost, optionally after 8.1-style normalization and with a domain threshold.

In [ ]:

def normalize_for_matching(text):
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.casefold()
    return "".join(ch for ch in text if ch.isalnum() or ch.isspace()).strip()


def best_match(query, candidates, threshold, normalize=False):
    q = normalize_for_matching(query) if normalize else query
    scored = []
    for cand in candidates:
        c = normalize_for_matching(cand) if normalize else cand
        dist, table, back = edit_distance(q, c)
        scored.append((dist, cand))
    scored.sort(key=lambda item: (item[0], item[1]))
    if scored[0][0] <= threshold:
        return scored[0][1], scored[0][0]
    return None, scored[0][0]

kitten_distance, kitten_table, kitten_back = edit_distance("kitten", "sitting")
assert kitten_distance == 3


## Dataset ladder: inline string pairs and retrieval dictionaries

In [ ]:

edit_ladder = [
    {
        "name": "D1 lesson cat cut",
        "queries": [("cat", ["cut", "dog", "cart"], "cut", 1)],
    },
    {
        "name": "D2 five clean typo pairs",
        "queries": [
            ("adress", ["address", "adjust", "artist"], "address", 2),
            ("recieve", ["receive", "recipe", "review"], "receive", 2),
            ("teh", ["the", "ten", "tech"], "the", 2),
            ("colour", ["color", "colon", "cooler"], "color", 2),
            ("kitten", ["sitting", "kitchen", "mitten"], "mitten", 2),
        ],
    },
    {
        "name": "D3 accent case threshold conflicts",
        "queries": [
            ("Cafe", ["Café", "cave", "safe"], "Café", 1),
            ("US", ["us", "USA", "UPS"], "us", 0),
            ("résumé", ["resume", "rescue", "assume"], "resume", 1),
            ("plan500", ["plan5000", "plan50", "plain500"], "plan50", 2),
            ("no", ["on", "now", "not"], "on", 2),
        ],
    },
    {
        "name": "D4 inline search-query dictionary",
        "queries": [
            ("campain manger", ["campaign manager", "campaign budget", "company manager"], "campaign manager", 3),
            ("invoice reciept", ["invoice receipt", "voice receipt", "invoice rejected"], "invoice receipt", 3),
            ("creativ library", ["creative library", "creator library", "creative history"], "creative library", 3),
            ("São invoice", ["sao invoice", "sales invoice", "sign invoice"], "sao invoice", 2),
            ("ad preview", ["ad preview", "ad previous", "add review"], "ad preview", 1),
        ],
    },
    {
        "name": "D5 longer entity names with misleading near matches",
        "queries": [
            ("Color Labs invoice", ["Colon Labs invoice", "Color Labs invoice", "Cold Labs invoice"], "Color Labs invoice", 2),
            ("Apple US campaign", ["apple us campaign", "Apple USA campaign", "Apple US campaign"], "Apple US campaign", 1),
            ("Cafe Rio São Paulo", ["Café Rio Sao Paulo", "Cafe Rico Sao Paulo", "Cafe Rio Santo Paulo"], "Café Rio Sao Paulo", 3),
            ("Plan 5000 not eligible", ["Plan 500 not eligible", "Plan 5000 eligible", "Plan 5000 not eligible"], "Plan 5000 not eligible", 2),
            ("Creator Marketplace", ["Creater Marketplace", "Creator Marketplaces", "Creator Marketplace"], "Creator Marketplace", 1),
        ],
    },
]

for rung in edit_ladder:
    print(rung["name"], "queries", len(rung["queries"]))
    print("sample", rung["queries"][0])


In [ ]:

edit_results = []
for index, rung in enumerate(edit_ladder, start=1):
    predictions = []
    expected = []
    for query, candidates, gold, threshold in rung["queries"]:
        pred, distance = best_match(query, candidates, threshold, normalize=True)
        predictions.append(pred)
        expected.append(gold)
    acc = exact_accuracy(predictions, expected)
    edit_results.append({"rung": index, "name": rung["name"], "accuracy": acc})

for row in edit_results:
    print(row["rung"], row["name"], f"accuracy={row['accuracy']:.3f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for idx, rung in enumerate(edit_ladder):
    query, candidates, gold, threshold = rung["queries"][0]
    pred, distance = best_match(query, candidates, threshold, normalize=True)
    dist, table, back = edit_distance(normalize_for_matching(query), normalize_for_matching(gold))
    show_matrix(axes[0, idx], table, f"D{idx + 1} DP", list("#" + normalize_for_matching(gold)), list("#" + normalize_for_matching(query)))

x = [row["rung"] for row in edit_results]
y = [row["accuracy"] for row in edit_results]
axes[1, 0].plot(x, y, marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_title("Accuracy vs harder strings")
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("match accuracy")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on D5: near spelling is not semantic identity

In [ ]:

query = "Color Labs invoice"
candidates = ["Colon Labs invoice", "Cold Labs invoice"]
unsafe_pred, unsafe_dist = best_match(query, candidates, threshold=2, normalize=True)
safe_candidates = ["Color Labs invoice", "Colon Labs invoice", "Cold Labs invoice"]
safe_pred, safe_dist = best_match(query, safe_candidates, threshold=0, normalize=False)

print("unsafe radius-2 result", unsafe_pred, unsafe_dist)
print("safe entity result", safe_pred, safe_dist)
print("colon distance", edit_distance("color", "colon")[0])


## Evaluate it + Practice

- Metric: match accuracy at a chosen edit threshold on each rung
- No-skill baseline: exact string equality, which misses color/colour and OCR typos
- Cheap sanity check: cat to cut must produce final distance 1 and boundary column [0,1,2,3]
- Ablation: remove normalization or use one global threshold and D3/D5 decisions change
- Failure signals: false entity matches, zero-cost missing prefixes, or thresholds that accept unrelated words

### Practice

**Practice:** Change substitution cost to 2 and inspect the cat to cut table.

**Practice:** Add a D5 company name where the nearest spelling is the wrong company.